# 09 - Query Handling and Adaptive RAG
## Shakespeare Evidence Desk

- **Goal:** decide, *per query*, how to prepare and route a question **before** the fixed stage 05-08 pipeline runs — normalizing vague phrasing, decomposing multi-part or cross-play questions into sub-queries, broadening retrieval to cover every work the question touches, and adding *bounded* adaptive behavior — so the single frozen baseline from stage 08 becomes query-aware without ever weakening grounding or abstention.
- **Experiment axis:** the query-handling policy that sits in front of retrieval — passthrough (the raw query) vs. single-query rewriting vs. decomposition with multi-query fusion vs. an adaptive router with a bounded retry (retrieval, reranking, the context builder, and the grounding contract are held fixed; only what we do to the *query* changes).
- **Loads from Step 08:** `.artifacts/generation-handoff.json` (the strict grounding contract), replayed on top of the retrieval, reranking, index, chunk, and context handoffs to rebuild each question's end-to-end answer.
- **Exports:** `.artifacts/query-handoff.json` — the selected query-handling policy that stages 10-12 replay and that `production/retrieval/query_processor.py` implements.


## Why This Stage

*In general — what query handling is and why an adaptive Retrieval-Augmented Generation (RAG) system needs it.* Every stage from 05 to 08 assumed the question arrives in a form retrieval can use. It rarely does. A user asks "how do the ghosts change the two heroes?" and the fixed pipeline embeds that one string, retrieves one pool, and — because the sentence leans toward whichever play dominates the vector — often returns passages from *Hamlet* alone, leaving *Macbeth* uncited and the comparison half-answered. The retriever is only ever as good as the query handed to it, and the query is the one thing the previous stages never touched.

This stage owns that gap: *what we do to the question before retrieval runs, and how much work each question is worth.* A simple lookup ("who speaks 'To be, or not to be'?") needs nothing but a single search. A multi-part, cross-play, or ambiguous-name question needs to be **decomposed** into independent sub-queries so retrieval can cover every work it touches. A vaguely worded question benefits from being **rewritten** into the corpus's vocabulary. And when the strict generator abstains on a question we had good reason to think was answerable, one **bounded retry** with a broader search can recover it — but exactly one, because unbounded adaptive loops are where cost and latency explode and determinism dies.

The choice is *measured*, as always. More query handling means more model calls and more moving parts, so we keep the simplest policy that covers every work a question expects, never weakens abstention on out-of-corpus questions, keeps every answer cited, and stays inside a small per-query call budget. Crucially, grounding authority stays with the stage 08 generator: query handling may broaden *what we retrieve*, but it never fabricates an answer and never lets a router skip the evidence for an in-scope question.


## Conceptual Foundations

### Query transformation: rewriting and expansion

The raw user question is written for a human, not for a retriever. **Query transformation** reshapes it into something that matches the corpus better: *normalization* fixes vague or elliptical phrasing, and *expansion* adds the play names, character names, or near-synonyms the passages actually use. The intent must survive the rewrite — a transformation that quietly changes the question is worse than none, because it answers something the user did not ask.

### Decomposition and multi-query retrieval

Some questions carry several retrieval targets at once: a **cross-play** comparison (Hamlet *and* Macbeth), an **ambiguous name** that appears in several works (the three Antonios), or a multi-part ask. Sent whole, these collapse onto whichever target the embedding weights most heavily. **Decomposition** splits the question into independent sub-queries, each aimed at one work or entity; **multi-query retrieval** runs a search for each and fuses the pooled candidates with **Reciprocal Rank Fusion (RRF)** — the same rank-combination tool stage 05 used for hybrid search, reused here to combine *sub-query* pools instead of *retriever* pools. The payoff is **coverage**: every work the question touches earns a seat in the context.

### Routing: spending effort where it pays

Not every query deserves the same machinery. **Routing** is a cheap up-front classification that decides how much handling a question is worth — a simple lookup gets a single search, a complex one gets decomposition. In fuller systems a router may also send a trivial greeting straight to a direct reply with no retrieval, or flag an obviously out-of-scope question. Routing is an *optimization*, never an authority: in a grounded system the router must not be the thing that decides an answer is unsupported — that judgment belongs to the generator reading the evidence.

### Adaptive RAG and bounded correction

**Adaptive RAG** lets the pipeline react to its own results instead of running one fixed path. The most useful and safest form here is a **bounded retry**: if the strict generator abstains on a question the router judged answerable, broaden the search *once* and try again before accepting the abstention. "Once" is the entire discipline — a **corrective** or **agentic** loop that keeps re-querying until it is satisfied can be powerful but is unbounded in cost, latency, and behavior, and it is exactly the thing a measured system refuses to ship without a cap.

### Conversational follow-up resolution

In a multi-turn session, "and what about the other play?" only means something against the previous turn. **Follow-up resolution** rewrites such a query into a standalone one by resolving its references against the conversation history. Our evaluation set is single-turn, so we *explain* this technique but do not measure it — the rewrite machinery here is where it would attach.

### What query handling is *not* (clean stage boundaries)

- **Not retrieval (05), reranking (06), or context building (07):** it does not search, reorder, or assemble evidence; it decides *what query (or queries)* those fixed stages receive.
- **Not generation (08):** it never writes or judges an answer and never overrides abstention; it only shapes and routes the question feeding the frozen grounding contract.
- **Not evaluation (10):** it picks a query-handling policy on the development set; the systematic, held-out scoring of the end-to-end system is stage 10's job.

### Common misconceptions and failure modes

- **"Rewriting always helps retrieval."** A rewrite can drift the intent or over-specify, retrieving confidently for the wrong question. Intent preservation matters more than fluency.
- **"Decompose everything."** Splitting a simple question into sub-queries adds cost and can dilute precision by pulling in loosely related passages. Decomposition earns its keep only on genuinely multi-target questions.
- **"Let the router answer or abstain."** A router that skips retrieval for an in-scope question, or declares a question unanswerable on its own, moves the grounding decision away from the evidence — the single most dangerous mistake in this stage.
- **"More adaptivity is better."** Unbounded retrieval loops raise recall asymptotically while cost and latency run away and results stop being reproducible. Bounded beats clever.
- **"Multi-query is free recall."** Fusing many pools raises coverage but also lengthens the candidate list the budget must still cut down to the top passages — coverage gained upstream can be lost again at context building if the budget is tight.


## The Technique Landscape

We **fix** everything stage 05-08 decided — dense retrieval, the selected reranker, the `dedup-parent` context builder and its token budget, and the strict grounding contract — and vary only the **query-handling policy** in front of them. Four policies are compared head-to-head; the rest are explained so the choice is informed.

| Policy / technique | Role here | What it does | Strengths | Weaknesses |
|---|---|---|---|---|
| **Passthrough** | compared (control) | send the raw question to one retrieval, then generate | zero added calls; the stage 08 baseline | misses works on multi-target questions; no help for vague phrasing |
| **Rewrite** | compared | one Large Language Model (LLM) reformulation into corpus vocabulary, then one retrieval | cheap (one extra call); helps vague or under-specified queries | can drift the intent; does nothing for multi-target coverage |
| **Decompose + multi-query** | compared | split into sub-queries, retrieve each, fuse pools with Reciprocal Rank Fusion (RRF), generate once | best coverage on cross-play and ambiguous-name questions | extra calls per sub-query; can over-fragment simple questions |
| **Adaptive (route + bounded retry)** | compared | route by complexity (rewrite vs decompose), and retry once with a broader search if a likely-answerable question is abstained | spends effort where it pays; can recover a near-miss answer | the most calls; routing can mis-classify; retry adds latency |
| **Hypothetical Document Embeddings (HyDE)** | explained, not run | draft a hypothetical answer and embed *that* for retrieval | can help recall on sparse phrasing | an extra generation per query; risks embedding a hallucinated draft |
| **Step-back prompting** | explained, not run | ask a more general question first to retrieve background, then the specific one | good for reasoning-heavy asks | more calls; benefit is hard to measure on factual lookups |
| **Conversational follow-up resolution** | explained, not run | rewrite a context-dependent follow-up into a standalone query | essential for multi-turn chat | needs the conversation history our single-turn eval set lacks |
| **Unbounded agentic / corrective retrieval loop** | explained, rejected | keep re-querying and re-reading until satisfied | highest ceiling on recall | unbounded cost, latency, and non-determinism — the opposite of a measured baseline |

So four bounded policies are measured; the rest are explained to frame the trade-off.


## 1. Load the Previous Selection

- **Purpose:** start from the exported decisions, not hidden kernel state. Query handling sits *in front of* a frozen pipeline, so retrieval (stage 05), reranking (stage 06), the context builder (stage 07), and the grounding contract (stage 08) are all fixed inputs rebuilt here from their handoffs. This stage changes only the query.
- **What the code does:** locates the repository root; reads `generation-handoff.json` (the strict contract and its exact system prompt) and, through the upstream handoffs, rebuilds the stage 05-08 machine — Best Matching 25 (BM25), dense, and hybrid Reciprocal Rank Fusion (RRF) retrieval, the selected reranker, the `dedup-parent` context builder, the disk-cached query embedder, and the cached strict generator; generalizes context assembly to accept a **list** of query variants and fuse their pools with RRF; and loads **every** labeled dev question so both coverage and abstention can be measured.
- **Fails loudly, early:** if any upstream artifact is missing, run notebooks 04-08 first. If the index was built on offline-fallback vectors (`embedding_backend != 'hosted'`), the notebook **stops here** — adapting queries over fake retrieval proves nothing.
- **Expected result:** the frozen upstream configuration, the loaded grounding contract, the Azure deployments, and how many answerable and unanswerable dev questions were loaded. Embeddings and generations are served from the stage 03-08 disk caches, so a warm run bills nothing.


In [ ]:
import hashlib
import json
import math
import re
import time
from collections import defaultdict
from pathlib import Path

import tiktoken
from dotenv import dotenv_values
from azure.identity import DefaultAzureCredential, get_bearer_token_provider
from openai import AzureOpenAI

# Work from the repository root so the notebook and the package read the same .env and artifacts.
ROOT = Path.cwd().resolve()
if not (ROOT / '.env.example').is_file() and (ROOT.parent / '.env.example').is_file():
    ROOT = ROOT.parent
if not (ROOT / '.env.example').is_file():
    raise RuntimeError('Use the repository as the notebook working directory.')

ARTIFACTS = ROOT / '.artifacts'
generation_handoff_path = ARTIFACTS / 'generation-handoff.json'
context_handoff_path = ARTIFACTS / 'context-handoff.json'
index_path = ARTIFACTS / 'index.json'
retrieval_handoff_path = ARTIFACTS / 'retrieval-handoff.json'
reranking_handoff_path = ARTIFACTS / 'reranking-handoff.json'
chunks_path = ARTIFACTS / 'chunks.json'
questions_path = ARTIFACTS / 'eval-questions.json'
for required in (generation_handoff_path, context_handoff_path, index_path, retrieval_handoff_path, reranking_handoff_path):
    if not required.is_file():
        raise RuntimeError(f'Missing .artifacts/{required.name}. Run notebooks 04-08 first.')

generation_handoff = json.loads(generation_handoff_path.read_text(encoding='utf-8'))
context_handoff = json.loads(context_handoff_path.read_text(encoding='utf-8'))
index_payload = json.loads(index_path.read_text(encoding='utf-8'))
retrieval_handoff = json.loads(retrieval_handoff_path.read_text(encoding='utf-8'))
reranking_handoff = json.loads(reranking_handoff_path.read_text(encoding='utf-8'))
chunks_payload = json.loads(chunks_path.read_text(encoding='utf-8')) if chunks_path.is_file() else {'chunks': []}
evaluation_questions = json.loads(questions_path.read_text(encoding='utf-8')) if questions_path.is_file() else {'development': [], 'heldout': []}

index_type = index_payload['index_type']
metric = index_payload['metric']
stored_dims = index_payload['dims']
embedding_backend = index_payload.get('embedding_backend', 'unknown')
records = index_payload['records']

# Stop now if upstream vectors are offline-fallback: adapting queries over fake retrieval proves nothing.
if embedding_backend != 'hosted':
    raise RuntimeError(
        f"index.json was built on the '{embedding_backend}' backend, not hosted Azure. "
        "Re-run notebooks 03-08 against Azure before tuning query handling.")

# ---- fixed upstream decisions this stage sits in front of (never re-decided here) ----
SELECTED_METHOD = retrieval_handoff['selected_method']
RRF_K = retrieval_handoff.get('rrf_k') or 60
bm25_params = retrieval_handoff.get('bm25_params', {'k1': 1.5, 'b': 0.75})
SELECTED_RERANKER = reranking_handoff['selected_reranker']
RERANK_POOL = reranking_handoff.get('rerank_pool', 20)
TOP_K = reranking_handoff.get('top_k', 5)
LLM_SNIPPET_CHARS = reranking_handoff.get('llm_reranker', {}).get('snippet_chars', 320)

SELECTED_BUILDER = context_handoff['selected_builder']          # raw | dedup-budget | dedup-parent
CONTEXT_TOKEN_BUDGET = context_handoff['token_budget']
CONTEXT_POOL = context_handoff.get('candidate_pool', RERANK_POOL)
DEDUP_THRESHOLD = context_handoff.get('dedup', {}).get('threshold', 0.6)
PARENT_NEIGHBORS = context_handoff.get('parent_context', {}).get('neighbors_each_side', 1)
PARENT_MAX_TOKENS = context_handoff.get('parent_context', {}).get('max_tokens', 700)

# ---- the frozen stage 08 grounding contract this stage routes every query through ----
SELECTED_CONTRACT = generation_handoff['selected_contract']
GEN_SYSTEM_PROMPT = generation_handoff['system_prompt']
GEN_USES_CONTEXT = generation_handoff.get('uses_context', True)
MAX_COMPLETION_TOKENS = generation_handoff.get('max_completion_tokens', 2000)
ABSTAIN_SENTINEL = generation_handoff.get('abstention', {}).get('sentinel', 'INSUFFICIENT_CONTEXT')

vectors = [r['embedding'] for r in records]
N = len(vectors)
work_titles = {r['work_id']: r['work'] for r in records}
corpus_works = evaluation_questions.get('corpus_works') or sorted(set(work_titles.values()))
encoder = tiktoken.get_encoding('cl100k_base')   # the same tokenizer the chunker used in notebook 02


def count_tokens(text):
    return len(encoder.encode(text))


def dot(a, b):
    # Score between two unit vectors; equals cosine similarity. Higher = more relevant.
    return sum(x * y for x, y in zip(a, b))


def l2_normalize(vector):
    length = math.sqrt(sum(value * value for value in vector))
    return [value / length for value in vector] if length else vector


_WORD_PATTERN = re.compile(r"[a-z0-9']+")


def tokenize(text):
    return _WORD_PATTERN.findall(text.lower())


# ---- replay stage 04 dense search + stage 05 BM25/RRF so each query rebuilds its pool ----
def exact_search_rows(query_vector, candidate_rows=None):
    pool = range(N) if candidate_rows is None else candidate_rows
    return sorted(pool, key=lambda i: dot(query_vector, vectors[i]), reverse=True)


if index_type == 'ivf':
    ivf = index_payload['ivf']
    ivf_centroids = ivf['centroids']
    ivf_lists = {int(cluster): rows for cluster, rows in ivf['lists'].items()}
    nprobe = index_payload['params']['nprobe']

    def dense_search(query_vector, k, candidate_rows=None):
        nearest = sorted(range(len(ivf_centroids)),
                         key=lambda c: dot(query_vector, ivf_centroids[c]), reverse=True)[:nprobe]
        pool = [i for c in nearest for i in ivf_lists.get(c, ())]
        if candidate_rows is not None:
            allowed = set(candidate_rows)
            pool = [i for i in pool if i in allowed]
        return exact_search_rows(query_vector, pool)[:k]
else:
    def dense_search(query_vector, k, candidate_rows=None):
        return exact_search_rows(query_vector, candidate_rows)[:k]


class BM25:
    """The same transparent Okapi BM25 keyword index from notebook 05, reused to replay its pool."""

    def __init__(self, documents, term_saturation=1.5, length_strength=0.75):
        self.k1 = term_saturation
        self.b = length_strength
        self.tokenized_docs = [tokenize(text) for text in documents]
        self.doc_lengths = [len(tokens) for tokens in self.tokenized_docs]
        self.avg_len = (sum(self.doc_lengths) / len(self.tokenized_docs)) if self.tokenized_docs else 0.0
        self.term_counts_per_doc = []
        document_frequency = defaultdict(int)
        for tokens in self.tokenized_docs:
            term_counts = defaultdict(int)
            for term in tokens:
                term_counts[term] += 1
            self.term_counts_per_doc.append(term_counts)
            for term in term_counts:
                document_frequency[term] += 1
        total_documents = len(self.tokenized_docs)
        self.idf = {
            term: math.log(1 + (total_documents - doc_count + 0.5) / (doc_count + 0.5))
            for term, doc_count in document_frequency.items()
        }

    def search(self, query, top_k, candidate_rows=None):
        query_terms = [term for term in tokenize(query) if term in self.idf]
        candidate_indices = range(len(self.tokenized_docs)) if candidate_rows is None else candidate_rows
        scored_rows = []
        for row_index in candidate_indices:
            term_counts = self.term_counts_per_doc[row_index]
            doc_length = self.doc_lengths[row_index]
            score = 0.0
            for term in query_terms:
                term_count = term_counts.get(term, 0)
                if not term_count:
                    continue
                length_normalizer = self.k1 * (1 - self.b + self.b * doc_length / (self.avg_len or 1))
                score += self.idf[term] * (term_count * (self.k1 + 1)) / (term_count + length_normalizer)
            if score > 0:
                scored_rows.append((row_index, score))
        scored_rows.sort(key=lambda row_and_score: row_and_score[1], reverse=True)
        return [row_index for row_index, _ in scored_rows[:top_k]]


bm25 = BM25([record['text'] for record in records],
            term_saturation=bm25_params.get('k1', 1.5), length_strength=bm25_params.get('b', 0.75))


def reciprocal_rank_fusion(rankings, damping, top_k):
    fused_scores = defaultdict(float)
    for ranking in rankings:
        for rank, row_index in enumerate(ranking):
            fused_scores[row_index] += 1.0 / (damping + rank + 1)
    ranked_rows = sorted(fused_scores.items(), key=lambda row_and_score: row_and_score[1], reverse=True)
    return [row_index for row_index, _ in ranked_rows[:top_k]]


def retrieve_pool(query_text, query_vector, pool_size):
    """Replay the stage 05 selected method to produce one query's candidate pool."""
    if SELECTED_METHOD == 'bm25':
        return bm25.search(query_text, pool_size)
    if SELECTED_METHOD == 'dense':
        return dense_search(query_vector, pool_size)
    dense_ranking = dense_search(query_vector, pool_size)
    keyword_ranking = bm25.search(query_text, pool_size)
    return reciprocal_rank_fusion([dense_ranking, keyword_ranking], RRF_K, pool_size)


# ---- parent/neighbor lookup for the selected context builder ----
chunks_by_id = {c['chunk_id']: c for c in chunks_payload.get('chunks', [])}


def chunk_index(chunk_id):
    # chunk_id is 'work_id:strategy:index'; the trailing integer is the chunk's position within its work.
    tail = chunk_id.rsplit(':', 1)[-1]
    return int(tail) if tail.isdigit() else None


siblings_by_work = defaultdict(dict)
for chunk_id, chunk in chunks_by_id.items():
    position = chunk_index(chunk_id)
    if position is not None:
        siblings_by_work[chunk.get('work_id')][position] = chunk_id

# ---- Azure clients (keyless, Entra identity); same on-disk caches as notebooks 03-08 ----
config = {key: value for key, value in dotenv_values(ROOT / '.env').items() if value}
token_provider = get_bearer_token_provider(DefaultAzureCredential(), 'https://cognitiveservices.azure.com/.default')
client = AzureOpenAI(
    azure_endpoint=config['AZURE_OPENAI_ENDPOINT'],
    api_version=config['AZURE_OPENAI_API_VERSION'],
    azure_ad_token_provider=token_provider,
)
embedding_deployment = config['AZURE_OPENAI_EMBEDDING_DEPLOYMENT']
chat_deployment = config['AZURE_OPENAI_CHAT_DEPLOYMENT']

EMBED_CACHE = ARTIFACTS / 'embedding-cache'
GEN_CACHE = ARTIFACTS / 'generation-cache'
EMBED_CACHE.mkdir(parents=True, exist_ok=True)
GEN_CACHE.mkdir(parents=True, exist_ok=True)
embed_api_calls = 0
gen_api_calls = 0


def embed_query(text):
    """Embed one query, reusing the 03-08 on-disk cache; call Azure only on a cache miss."""
    global embed_api_calls
    cache_file = EMBED_CACHE / (hashlib.sha256(f'{embedding_deployment}|{text}'.encode('utf-8')).hexdigest() + '.json')
    if cache_file.is_file():
        full_vector = json.loads(cache_file.read_text())
    else:
        full_vector = client.embeddings.create(model=embedding_deployment, input=[text]).data[0].embedding
        cache_file.write_text(json.dumps(full_vector))
        embed_api_calls += 1
    return l2_normalize(full_vector[:stored_dims])


def apply_selected_reranker(query_text, query_vector, candidate_rows):
    """Reorder the pool exactly as stage 06 decided (this corpus selected 'none')."""
    if SELECTED_RERANKER == 'semantic':
        return sorted(candidate_rows, key=lambda row: dot(query_vector, vectors[row]), reverse=True)
    return list(candidate_rows)


# ---- the stage 07 context builder toolkit (dedup, parent expansion, citation assembly) ----
def token_set(text):
    return set(tokenize(text))


def jaccard(a, b):
    union = len(a | b)
    return len(a & b) / union if union else 0.0


def format_scene(scenes):
    if not scenes:
        return 'Unknown scene'
    act, scene = scenes[0]
    return f'Act {act}, Scene {scene}'


def parent_text(row):
    """Expand one chunk to its same-scene neighbors, overlap lines de-duplicated and the result token-capped."""
    record = records[row]
    work_id = record['work_id']
    position = chunk_index(record['chunk_id'])
    if position is None or work_id not in siblings_by_work:
        return record['text']
    pieces = []
    for offset in range(-PARENT_NEIGHBORS, PARENT_NEIGHBORS + 1):
        sibling_id = siblings_by_work[work_id].get(position + offset)
        sibling = chunks_by_id.get(sibling_id) if sibling_id else None
        if sibling and any(scene in record['scenes'] for scene in sibling.get('scenes', [])):
            pieces.append((position + offset, sibling['text']))
    pieces.sort()
    seen_lines, merged_lines = set(), []
    for _, text in pieces:
        for line in text.split('\n'):
            stripped = line.strip()
            if stripped and stripped in seen_lines:   # drop the overlap lines the chunker repeated
                continue
            if stripped:
                seen_lines.add(stripped)
            merged_lines.append(line)
    merged = '\n'.join(merged_lines).strip() or record['text']
    token_ids = encoder.encode(merged)
    return encoder.decode(token_ids[:PARENT_MAX_TOKENS]) if len(token_ids) > PARENT_MAX_TOKENS else merged


def passage_block(position, record, text):
    header = f"[{position}] {record['work']} - {format_scene(record['scenes'])}"
    if record['speakers']:
        header += f" ({', '.join(record['speakers'][:4])})"
    return f"{header}\n{text.strip()}"


def assemble(rows, texts):
    """Number the passages, build the context string, and attach a resolvable citation to each."""
    blocks, citations = [], []
    for position, (row, text) in enumerate(zip(rows, texts), start=1):
        record = records[row]
        blocks.append(passage_block(position, record, text))
        citations.append({
            'marker': position,
            'chunk_id': record['chunk_id'],
            'work': record['work'],
            'work_id': record['work_id'],
            'scene': format_scene(record['scenes']),
            'speakers': record['speakers'],
        })
    return {'rows': rows, 'context': '\n\n'.join(blocks), 'citations': citations}


def build_raw(ordered_rows):
    chosen = list(ordered_rows[:TOP_K])
    return assemble(chosen, [records[row]['text'] for row in chosen])


def _pack(ordered_rows, expand):
    kept, kept_signatures, texts, used_tokens = [], [], [], 0
    for row in ordered_rows:
        signature = token_set(records[row]['text'])
        if any(jaccard(signature, previous) >= DEDUP_THRESHOLD for previous in kept_signatures):
            continue
        text = parent_text(row) if expand else records[row]['text']
        block_tokens = count_tokens(passage_block(len(kept) + 1, records[row], text)) + 2
        if used_tokens + block_tokens > CONTEXT_TOKEN_BUDGET:
            continue
        kept.append(row)
        kept_signatures.append(signature)
        texts.append(text)
        used_tokens += block_tokens
    return assemble(kept, texts)


BUILDERS = {
    'raw': build_raw,
    'dedup-budget': lambda rows: _pack(rows, expand=False),
    'dedup-parent': lambda rows: _pack(rows, expand=True),
}
build_context = BUILDERS[SELECTED_BUILDER]


def build_context_for_queries(primary_query, variants, pool_size):
    """Retrieve a pool per query variant, fuse them with RRF, then rerank and assemble one cited context.

    A single variant reduces to the stage 07/08 path; several variants broaden coverage by fusing pools.
    """
    rankings = []
    for variant in variants:
        variant_vector = embed_query(variant)
        rankings.append(retrieve_pool(variant, variant_vector, pool_size))
    fused = rankings[0] if len(rankings) == 1 else reciprocal_rank_fusion(rankings, RRF_K, pool_size)
    primary_vector = embed_query(primary_query)
    ordered = apply_selected_reranker(primary_query, primary_vector, fused)
    return build_context(ordered)


def generate(question_text, context_text):
    """One grounded answer under the frozen stage 08 contract, cached to disk and replayed on a miss."""
    global gen_api_calls
    user_message = (f'Context:\n{context_text}\n\nQuestion: {question_text}'
                    if GEN_USES_CONTEXT else f'Question: {question_text}')
    cache_key = hashlib.sha256(
        f"{chat_deployment}|{SELECTED_CONTRACT}|{MAX_COMPLETION_TOKENS}|{user_message}".encode('utf-8')).hexdigest()
    cache_file = GEN_CACHE / (cache_key + '.json')
    if cache_file.is_file():
        cached = json.loads(cache_file.read_text())
        return cached['answer'], cached.get('latency_ms', 0.0)
    started = time.perf_counter()
    response = client.chat.completions.create(
        model=chat_deployment,
        max_completion_tokens=MAX_COMPLETION_TOKENS,
        messages=[{'role': 'system', 'content': GEN_SYSTEM_PROMPT},
                  {'role': 'user', 'content': user_message}],
    )
    latency_ms = (time.perf_counter() - started) * 1000
    answer = (response.choices[0].message.content or '').strip()
    cache_file.write_text(json.dumps({'answer': answer, 'latency_ms': latency_ms}))
    gen_api_calls += 1
    return answer, latency_ms


# ---- load EVERY dev question (answerable + unanswerable) so coverage and abstention both apply ----
dev_questions = list(evaluation_questions.get('development', []))
answerable_count = sum(1 for q in dev_questions if q.get('answerable'))
unanswerable_count = len(dev_questions) - answerable_count

print(f'Frozen upstream: retrieval={SELECTED_METHOD}, reranker={SELECTED_RERANKER}, top-k={TOP_K}, '
      f'builder={SELECTED_BUILDER}, budget={CONTEXT_TOKEN_BUDGET} tokens.')
print(f'Grounding contract: {SELECTED_CONTRACT} | abstains via "{ABSTAIN_SENTINEL}" | budget={MAX_COMPLETION_TOKENS} tokens.')
print(f'Azure deployments: chat={chat_deployment}, embedding={embedding_deployment}.')
print(f'Corpus: {len(corpus_works)} plays. Loaded {len(dev_questions)} dev question(s): '
      f'{answerable_count} answerable, {unanswerable_count} unanswerable.')


## 2. Explain the Alternatives

A query-handling policy is **good** when it gets every work a question touches into the retrieved context, keeps the strict generator's abstention and citations intact, and spends as few extra model calls as the question actually needs. We **fix everything downstream** (retrieval, reranker, context builder, and the grounding contract) and vary only the **policy** applied to the query before retrieval.

### What we actually compare vs. what we only explain

| Approach | Role here | Why |
|---|---|---|
| **Passthrough** | compared (control) | the raw query feeds one retrieval — the stage 08 baseline every other policy must beat; it is the honest yardstick for what query handling actually buys |
| **Rewrite** | compared | one reformulation into the corpus's vocabulary before a single retrieval; isolates the value of *normalization* without any fan-out |
| **Decompose + multi-query** | compared | split multi-target questions into sub-queries, retrieve each, fuse the pools; isolates the value of *coverage* on cross-play and ambiguous-name questions |
| **Adaptive (route + bounded retry)** | compared | classify complexity to pick rewrite vs decompose, and retry once with a broader search if a likely-answerable question is abstained; tests whether *reacting* to results is worth the extra calls |
| **HyDE / step-back prompting** | explained, not run | reshape the embedding target with a drafted answer or a more general question; extra generations whose benefit is better judged by the stage 10 evaluator |
| **Conversational follow-up resolution** | explained, not run | resolve references against prior turns; our eval set is single-turn, so there is nothing to resolve against |
| **Unbounded agentic loop** | explained, rejected | re-query until satisfied; unbounded cost and non-determinism disqualify it as a measured baseline |

So four bounded policies are compared head-to-head; the rest are explained so the choice makes sense.

### How each compared policy actually works (a line or two)

- **Passthrough** — embed the raw question, retrieve one pool, build the context, generate. No reformulation, no fan-out. It makes visible exactly where a single query *fails to cover* a multi-work question.
- **Rewrite** — one Large Language Model (LLM) call rewrites the question into corpus vocabulary (naming likely plays and characters), then a single retrieval runs on the rewrite. Cheap; helps vague phrasing; does nothing structural for multi-target coverage.
- **Decompose + multi-query** — one LLM call splits the question into up to three independent sub-queries, each retrieves its own pool, and the pools are fused with **RRF** before context building. This is the policy built to cover *every* work a comparison or ambiguous name spans.
- **Adaptive** — one LLM call routes the question: *simple* gets a rewrite, *multi* gets decomposition. Then, if the strict generator abstains on a question the router judged plausibly in-corpus, the search is broadened **once** and generation retried before the abstention is accepted.

### Why this is the experiment we run

The stage exists to make the fixed baseline *query-aware* without breaking it. Query handling has three measurable failure modes: missing a work the question needs (coverage), drifting the intent or weakening abstention (safety), and spending calls a question did not warrant (cost). Passthrough versus rewrite isolates whether normalization alone helps; passthrough versus decompose isolates coverage on multi-target questions; decompose versus adaptive isolates whether reacting to an abstention is worth its extra calls. The sub-query cap, the single-retry bound, and the RRF damping are bounded settings, not competing arms.


## 3. Run Controlled Experiments

- **Fixed downstream:** retrieval method, reranker, context builder, token budget, and the strict grounding contract are all frozen from the handoffs. Every policy answers through the *same* generator; only the query (or queries) feeding retrieval changes, so any difference in coverage or answers is the policy, not the pipeline.
- **What the code does:** defines three cached query-handling operations — `rewrite_query` (one reformulation into corpus vocabulary), `decompose_query` (split into up to three sub-queries), and `route_query` (classify complexity and plausible in-corpus-ness) — each cached to disk by (deployment, operation, query); defines the four policies (passthrough, rewrite, decompose, adaptive) as thin controllers over those operations plus the frozen `build_context_for_queries` and `generate`; and runs every policy over every dev question, recording the assembled context, the answer, the latency, and the per-query **added model calls** over the passthrough baseline.
- **Bounded by design:** decomposition is capped at three sub-queries; the adaptive policy adds at most one broadening **retry** after an abstention, and only when the router judged the question plausibly in-corpus. The generator — never the router — decides whether the evidence supports an answer.
- **LLM calls:** reformulations and generations are cached to disk, so a warm run makes zero live calls; the recorded call counts are the *structural* cost of each policy, independent of caching.
- **Expected result:** the four registered policies, the number of answers produced per policy, the live-vs-cached call counts, and confirmation that every policy routed through the identical frozen generator.


In [ ]:
MAX_SUBQUERIES = 3          # cap decomposition; multi-target questions here span at most three works
RETRY_POOL_MULTIPLIER = 2   # the single adaptive retry widens the candidate pool, not the number of retries

_WORKS_SENTENCE = ', '.join(corpus_works)

REWRITE_SYSTEM = (
    'You rewrite search queries for a Shakespeare evidence desk whose corpus is exactly these ten plays: '
    f'{_WORKS_SENTENCE}. '
    'Rewrite the user question into a single search query phrased in the vocabulary of the plays: resolve vague '
    'references and name the most likely play(s) and character(s). Preserve the original intent exactly and do not '
    'answer the question. Return a JSON object {"query": "..."}.'
)

DECOMPOSE_SYSTEM = (
    'You split questions for a Shakespeare evidence desk whose corpus is exactly these ten plays: '
    f'{_WORKS_SENTENCE}. '
    'If the question targets several plays, characters, or parts at once (a comparison, an ambiguous name that '
    'appears in multiple plays, or a multi-part ask), break it into independent sub-questions, each aimed at ONE '
    'play or entity. If it already targets a single thing, return it unchanged as the only sub-question. Return at '
    f'most {MAX_SUBQUERIES} sub-questions, preserve intent, and do not answer. '
    'Return a JSON object {"subqueries": ["...", "..."]}.'
)

ROUTE_SYSTEM = (
    'You route questions for a Shakespeare evidence desk whose corpus is exactly these ten plays: '
    f'{_WORKS_SENTENCE}. '
    'Classify the question. "complexity" is "multi" if answering it well needs evidence from more than one play or '
    'entity (a comparison, an ambiguous name, or a multi-part ask), otherwise "simple". "likely_in_corpus" is true '
    'if the question could plausibly be answered from these ten plays. Do not answer the question. '
    'Return a JSON object {"complexity": "simple|multi", "likely_in_corpus": true|false}.'
)

QUERY_CACHE = ARTIFACTS / 'query-cache'
QUERY_CACHE.mkdir(parents=True, exist_ok=True)
query_api_calls = 0


def _chat_json(system_prompt, user_text, operation):
    """One JSON chat call for a query-handling operation, cached to disk by (deployment, operation, query)."""
    global query_api_calls
    cache_key = hashlib.sha256(f'{chat_deployment}|{operation}|{user_text}'.encode('utf-8')).hexdigest()
    cache_file = QUERY_CACHE / (cache_key + '.json')
    if cache_file.is_file():
        return json.loads(cache_file.read_text())
    response = client.chat.completions.create(
        model=chat_deployment,
        response_format={'type': 'json_object'},
        messages=[{'role': 'system', 'content': system_prompt},
                  {'role': 'user', 'content': user_text}],
    )
    parsed = json.loads(re.search(r'\{.*\}', response.choices[0].message.content or '{}', re.DOTALL).group(0))
    cache_file.write_text(json.dumps(parsed))
    query_api_calls += 1
    return parsed


def rewrite_query(question_text):
    rewritten = (_chat_json(REWRITE_SYSTEM, question_text, 'rewrite').get('query') or '').strip()
    return rewritten or question_text   # never let a blank rewrite drop the question


def decompose_query(question_text):
    out = _chat_json(DECOMPOSE_SYSTEM, question_text, 'decompose')
    subs = [str(sub).strip() for sub in out.get('subqueries', []) if str(sub).strip()]
    return subs[:MAX_SUBQUERIES] or [question_text]


def route_query(question_text):
    out = _chat_json(ROUTE_SYSTEM, question_text, 'route')
    complexity = out.get('complexity') if out.get('complexity') in ('simple', 'multi') else 'simple'
    return {'complexity': complexity, 'likely_in_corpus': bool(out.get('likely_in_corpus', True))}


def is_abstention(answer):
    return ABSTAIN_SENTINEL in answer.upper()


def run_query_policy(policy, question_text):
    """Apply one query-handling policy end to end and record its context, answer, latency, and call cost."""
    reform_calls, gen_calls, retried = 0, 1, False
    route = None
    if policy == 'passthrough':
        primary, variants = question_text, [question_text]
    elif policy == 'rewrite':
        primary = rewrite_query(question_text)
        variants = [primary]
        reform_calls = 1
    elif policy == 'decompose':
        variants = decompose_query(question_text)
        primary = variants[0]
        reform_calls = 1
    else:  # adaptive: route, then rewrite or decompose by complexity
        route = route_query(question_text)
        reform_calls = 1
        if route['complexity'] == 'multi':
            variants = decompose_query(question_text)
            primary = variants[0]
        else:
            primary = rewrite_query(question_text)
            variants = [primary]
        reform_calls += 1

    built = build_context_for_queries(primary, variants, CONTEXT_POOL)
    answer, latency = generate(question_text, built['context'])   # always answer the ORIGINAL question

    # Bounded adaptive correction: one broadening retry when a likely-answerable question is abstained.
    if policy == 'adaptive' and is_abstention(answer) and route['likely_in_corpus']:
        expansion = rewrite_query(question_text)
        reform_calls += 1
        variants = list(dict.fromkeys(variants + [expansion]))
        built = build_context_for_queries(primary, variants, CONTEXT_POOL * RETRY_POOL_MULTIPLIER)
        answer, retry_latency = generate(question_text, built['context'])
        latency += retry_latency
        gen_calls += 1
        retried = True

    embed_calls = len(variants)
    # Passthrough baseline costs 0 reformulations + 1 embedding + 1 generation; report the delta over that.
    added_calls = (reform_calls + embed_calls + gen_calls) - 2
    return {
        'context': built['context'],
        'citations': built['citations'],
        'answer': answer,
        'latency_ms': latency,
        'variants': variants,
        'added_calls': added_calls,
        'retried': retried,
    }


POLICIES = ['passthrough', 'rewrite', 'decompose', 'adaptive']
policy_runs = {policy: [] for policy in POLICIES}
for policy in POLICIES:
    for question in dev_questions:
        record = run_query_policy(policy, question['question'])
        record.update({
            'id': question.get('id'),
            'question': question['question'],
            'answerable': bool(question.get('answerable')),
            'expected_works': question.get('expected_works', []),
        })
        policy_runs[policy].append(record)

print(f'Registered policies: {", ".join(POLICIES)}.')
print(f'Produced {len(dev_questions)} answer(s) per policy ({len(POLICIES) * len(dev_questions)} total).')
print(f'Live calls this run: {query_api_calls} query-handling, {embed_api_calls} embedding, {gen_api_calls} generation '
      '(rest served from disk caches).')
print('Every policy answered the original question through the identical frozen stage 08 generator.')


## Metrics in This Stage — What Each Number Means

Section 4 runs the four policies over the same dev questions and grades each on coverage, safety, and cost. The selection rule keeps the simplest policy that covers every work a question expects, never weakens abstention or citations, and stays inside a small added-call budget.

| Metric | What it measures | How it is computed | What it signifies |
|---|---|---|---|
| Work coverage | whether retrieval reached every work the question needs | share of answerable questions whose assembled context cites at least one passage from every expected work | The lever this stage pulls — decomposition should raise it on multi-target questions; higher is better. |
| Answer rate | whether the system answers the questions it should | share of answerable questions the generator did not abstain on | Over-refusal and recovery guard; a bounded retry can lift it. Higher is better. |
| Abstention accuracy | whether the system still refuses the unanswerable | share of unanswerable questions answered with the refusal sentinel | Primary safety constraint — query handling must not erode it; must stay 1.0. |
| Citation validity | whether every answered claim is traceable | share of answered questions whose bracketed markers all resolve to a passage in that context, with at least one citation | Grounding constraint — fusing more pools must not break citations; must stay 1.0. |
| Grounding (lexical) | how much of the answer is drawn from the context | mean fraction of an answer's content words that appear in its context word set | Faithfulness proxy, pending the stage 10 judge; higher is better. |
| Work relevance | whether the answer names the expected work(s) | share of answered answerable questions whose answer mentions every expected work title | Correctness proxy at the corpus level; higher is better. |
| Added model calls | the extra cost the policy spends per query | mean of (reformulation + embedding + generation calls) minus the passthrough baseline of two | Cost of adaptivity; lower is better and breaks ties between policies that match on quality. |
| p95 latency (ms) | the per-query wall-clock cost | 95th percentile of the per-question end-to-end time | Reflects added reformulation, extra retrievals, and any retry; lower is better. |


## 4. Measure and Inspect

- **The lenses:** **work coverage** (did retrieval reach every work the question needs), **answer rate** and **abstention accuracy** (does it answer the answerable and still refuse the unanswerable), **citation validity** and **grounding** (is each answer traceable and drawn from the evidence), **work relevance** (does it name the right play), and **added calls / latency** (cost).
- **Work coverage is the lever this stage pulls.** For each answerable question we check whether the assembled context cites at least one passage from *every* expected work. Passthrough tends to cover single-work questions but miss a side of a cross-play comparison or a name that spans several plays; decomposition exists to close exactly that gap, and this metric is where it shows up.
- **Abstention accuracy and citation validity are hard constraints, carried over from stage 08.** Query handling is only allowed to change *what we retrieve*; it must not let the system answer an out-of-corpus question or produce an answer whose citations do not resolve. Any policy that erodes either is disqualified regardless of its coverage.
- **Added model calls is the cost axis.** Every reformulation, every extra sub-query embedding, and any retry is counted against the passthrough baseline of two calls. A policy that matches another on coverage and safety but spends more is the worse choice.
- **A worked comparison:** we print the decomposition's sub-queries for the cross-play ghost question and the set of works each policy actually got into the context, so the coverage difference is visible by eye rather than only in a table.
- **Honest limit:** the labeled dev set is tiny and single-turn, and grounding is a lexical proxy, so these numbers pre-screen policies rather than certify end-to-end quality. The systematic, held-out judge is stage 10.


In [ ]:
_STOPWORDS = {
    'the', 'a', 'an', 'and', 'or', 'but', 'of', 'to', 'in', 'on', 'for', 'with', 'as', 'by', 'at',
    'is', 'are', 'was', 'were', 'be', 'been', 'being', 'it', 'its', 'this', 'that', 'these', 'those',
    'he', 'she', 'they', 'his', 'her', 'their', 'who', 'what', 'which', 'how', 'does', 'do', 'did',
    'from', 'not', 'no', 'has', 'have', 'had', 'about', 'into', 'than', 'then', 'there', 'here',
}
_CITATION_PATTERN = re.compile(r'\[(\d+)\]')


def cited_markers(answer):
    return [int(marker) for marker in _CITATION_PATTERN.findall(answer)]


def citations_resolve(answer, citations):
    # Valid when the answer cites at least one passage and every cited number exists in this context.
    markers = cited_markers(answer)
    valid_markers = {citation['marker'] for citation in citations}
    return bool(markers) and all(marker in valid_markers for marker in markers)


def context_covers_works(citations, expected_works):
    # Coverage: the assembled context contains at least one passage from every work the question expects.
    covered = {citation['work'] for citation in citations}
    return all(work in covered for work in expected_works) if expected_works else True


def grounding_overlap(answer, context_text):
    # Lexical faithfulness proxy: share of the answer's content words that appear in the context.
    answer_terms = {term for term in tokenize(_CITATION_PATTERN.sub(' ', answer)) if term not in _STOPWORDS}
    if not answer_terms:
        return 0.0
    context_terms = set(tokenize(context_text))
    return len(answer_terms & context_terms) / len(answer_terms)


def names_expected_works(answer, expected_works):
    lowered = answer.lower()
    return all(str(work).lower() in lowered for work in expected_works) if expected_works else True


def evaluate_policy(runs):
    """Score one policy: coverage, answer rate, abstention, citations, grounding, relevance, and cost."""
    answerable = [r for r in runs if r['answerable']]
    unanswerable = [r for r in runs if not r['answerable']]
    abstained_unanswerable = sum(1 for r in unanswerable if is_abstention(r['answer']))
    answered = [r for r in answerable if not is_abstention(r['answer'])]
    covered = sum(1 for r in answerable if context_covers_works(r['citations'], r['expected_works']))
    citation_valid = sum(1 for r in answered if citations_resolve(r['answer'], r['citations']))
    grounding = [grounding_overlap(r['answer'], r['context']) for r in answered]
    relevance = sum(1 for r in answered if names_expected_works(r['answer'], r['expected_works']))
    added = [r['added_calls'] for r in runs]
    latencies = sorted(r['latency_ms'] for r in runs)
    p95 = latencies[min(len(latencies) - 1, int(0.95 * len(latencies)))] if latencies else 0.0
    return {
        'work_coverage': covered / max(len(answerable), 1),
        'answer_rate': len(answered) / max(len(answerable), 1),
        'abstention_accuracy': abstained_unanswerable / max(len(unanswerable), 1),
        'citation_validity': (citation_valid / len(answered)) if answered else 0.0,
        'grounding': (sum(grounding) / len(grounding)) if grounding else 0.0,
        'work_relevance': (relevance / len(answered)) if answered else 0.0,
        'added_calls': sum(added) / max(len(added), 1),
        'p95_latency_ms': p95,
    }


policy_metrics = {policy: evaluate_policy(runs) for policy, runs in policy_runs.items()}

print(f'Measured over {len(dev_questions)} dev question(s): '
      f'{answerable_count} answerable, {unanswerable_count} unanswerable.\n')
header = ('policy', 'cover', 'answer', 'abstain', 'cite', 'ground', 'workrel', 'addcalls', 'p95 ms')
print('{:<12} {:>6} {:>7} {:>8} {:>6} {:>7} {:>8} {:>9} {:>8}'.format(*header))
for name, m in policy_metrics.items():
    print('{:<12} {:>6.0%} {:>7.0%} {:>8.0%} {:>6.0%} {:>7.0%} {:>8.0%} {:>9.1f} {:>8.1f}'.format(
        name, m['work_coverage'], m['answer_rate'], m['abstention_accuracy'], m['citation_validity'],
        m['grounding'], m['work_relevance'], m['added_calls'], m['p95_latency_ms']))

# Worked example: show how decomposition broadens coverage on a cross-play question, visible by eye.
cross = next((r for r in policy_runs['decompose'] if len(r['expected_works']) > 1), None)
if cross:
    base = next((r for r in policy_runs['passthrough'] if r['id'] == cross['id']), None)
    print(f'\nCross-play question - "{cross["question"][:70]}..."')
    print(f'  expected works: {", ".join(cross["expected_works"])}')
    print('  decomposed sub-queries:')
    for sub in cross['variants']:
        print(f'    - {sub}')
    print(f'  works covered | passthrough: {sorted({c["work"] for c in base["citations"]}) if base else "n/a"}')
    print(f'  works covered | decompose:   {sorted({c["work"] for c in cross["citations"]})}')


## 5. Justify the Selection

- **Decision rule (data-driven, deterministic):** first keep only **feasible** policies — those that preserve **perfect abstention accuracy** (never answer an out-of-corpus question) and **full citation validity** (every answer still traces to a passage). Among the feasible, prefer **higher work coverage**, then **higher answer rate**, then **higher grounding**, then **fewer added calls**, then the **simpler** policy. The rule is applied to the measured numbers, not asserted.
- **Why abstention and citations gate everything:** query handling is only permitted to change *what we retrieve*. If broadening the search ever let the system answer an unanswerable question or produced an answer whose citations do not resolve, it broke the very guarantees stage 08 established — so those two constraints filter the candidates before any trade-off is weighed.
- **Why coverage leads and cost breaks ties:** the reason this stage exists is that a single query misses works on multi-target questions. Among safe, traceable policies, the best one covers the most expected works and answers the most answerable questions; when two tie, the one that spends fewer extra calls — and failing that, the simpler policy — is the honest choice.
- **Honest, data-dependent outcome (no pre-ordained winner):** passthrough is free but tends to under-cover cross-play and ambiguous-name questions. Rewrite helps vague phrasing but rarely fixes coverage. Decomposition is built to cover every work, at the cost of extra sub-query calls. Adaptive matches decomposition's coverage and can recover an abstained-but-answerable question, but spends the most — so it only wins if the numbers show it recovering coverage or answers the simpler policies miss. We report whichever the measured numbers choose.
- **Expected result:** the selected policy with the numbers that justify it, and each rejected alternative annotated with its measured reason.


In [ ]:
SIMPLICITY_ORDER = {'passthrough': 0, 'rewrite': 1, 'decompose': 2, 'adaptive': 3}   # tie-break; lower = simpler


def is_feasible(name):
    m = policy_metrics[name]
    # Hard constraints carried over from stage 08: never answer the unanswerable, and cite every answer given.
    return m['abstention_accuracy'] >= 1.0 - 1e-9 and m['citation_validity'] >= 1.0 - 1e-9


def selection_key(name):
    m = policy_metrics[name]
    # Among feasible: more coverage, then more answers, then more grounding, then fewer calls, then simpler.
    return (-m['work_coverage'], -m['answer_rate'], -m['grounding'], m['added_calls'], SIMPLICITY_ORDER.get(name, 99))


feasible = [name for name in policy_metrics if is_feasible(name)]
if feasible:
    selected_policy = min(feasible, key=selection_key)
else:
    # Nothing preserved the stage 08 guarantees: fall back to the safest and flag it loudly.
    selected_policy = max(policy_metrics,
                          key=lambda n: (policy_metrics[n]['abstention_accuracy'],
                                         policy_metrics[n]['citation_validity']))

selected = policy_metrics[selected_policy]
print(f'Selected policy: {selected_policy}')
print(f"  work coverage / answer rate:  {selected['work_coverage']:.0%} / {selected['answer_rate']:.0%}")
print(f"  abstention / citation:        {selected['abstention_accuracy']:.0%} / {selected['citation_validity']:.0%}")
print(f"  grounding / work relevance:   {selected['grounding']:.0%} / {selected['work_relevance']:.0%}")
print(f"  added calls / p95 latency:    {selected['added_calls']:.1f} / {selected['p95_latency_ms']:.0f} ms")
print('\nAlternatives:')
for name, m in policy_metrics.items():
    if name == selected_policy:
        continue
    if m['abstention_accuracy'] < 1.0 - 1e-9:
        reason = f"answers {1 - m['abstention_accuracy']:.0%} of out-of-corpus questions - disqualified"
    elif m['citation_validity'] < 1.0 - 1e-9:
        reason = f"only {m['citation_validity']:.0%} of answers cite a resolvable passage - disqualified"
    elif m['work_coverage'] < selected['work_coverage'] - 1e-9:
        reason = f"lower coverage ({m['work_coverage']:.0%} vs {selected['work_coverage']:.0%})"
    elif m['answer_rate'] < selected['answer_rate'] - 1e-9:
        reason = f"answers fewer ({m['answer_rate']:.0%} vs {selected['answer_rate']:.0%})"
    elif m['added_calls'] > selected['added_calls'] + 1e-9:
        reason = f"same quality but costs more ({m['added_calls']:.1f} vs {selected['added_calls']:.1f} added calls)"
    else:
        reason = 'feasible but not simpler than the selection'
    print(f"  {name}: cover={m['work_coverage']:.0%} answer={m['answer_rate']:.0%} abstain={m['abstention_accuracy']:.0%} "
          f"cite={m['citation_validity']:.0%} calls={m['added_calls']:.1f} - {reason}")


## 6. Export the Handoff

- **What gets exported:**
  - `.artifacts/query-handoff.json` — the decision record stages 10-12 load: the selected query-handling policy (`passthrough` / `rewrite` / `decompose` / `adaptive`), the exact parameters that reproduce it (whether it rewrites, whether it decomposes and the sub-query cap, the fusion method and Reciprocal Rank Fusion (RRF) damping, the candidate pool, and the retry budget and pool multiplier), the reformulation and routing system prompts, a back-reference to the frozen stage 05-08 pipeline it sits in front of, and the full policy comparison (coverage, answer rate, abstention, citations, grounding, relevance, cost).
- **Why a config handoff, not an answer dump:** the pipeline reproduces every answer from the upstream handoffs plus this policy. The handoff is that reproducible configuration plus the evidence for it, so stages 10-12 can replay or evaluate any answer without re-deciding anything.
- **Production target:** `production/retrieval/query_processor.py` implements `process_query(query)` to return `{'normalized', 'variants', 'embedding'}` — the normalized (rewritten or raw) query, the retrieval variants (sub-queries or the single query), and the embedding of the normalized query — exactly the shaping the selected policy performs before retrieval.
- **Hygiene:** `.artifacts/` is gitignored and notebook outputs are stripped on commit — do not commit the handoff or the query cache.
- **Expected result:** the handoff written, with the selected policy and its parameters printed.


In [ ]:
query_handoff_path = ARTIFACTS / 'query-handoff.json'

# The selected policy's reproducible parameters: what process_query must do before retrieval.
POLICY_PARAMETERS = {
    'passthrough': {'rewrite': False, 'decompose': False, 'route': False, 'retry_budget': 0},
    'rewrite': {'rewrite': True, 'decompose': False, 'route': False, 'retry_budget': 0},
    'decompose': {'rewrite': False, 'decompose': True, 'route': False, 'retry_budget': 0},
    'adaptive': {'rewrite': True, 'decompose': True, 'route': True, 'retry_budget': 1},
}

# query-handoff.json: the query-handling policy stages 10-12 replay in front of the frozen stage 05-08 pipeline,
# plus the measured comparison that justified it. The upstream handoffs hold everything retrieval and generation need.
query_handoff = {
    'stage': '09-query-handling-and-adaptive-rag',
    'selected_policy': selected_policy,                    # passthrough | rewrite | decompose | adaptive
    'policy_parameters': {
        **POLICY_PARAMETERS[selected_policy],
        'max_subqueries': MAX_SUBQUERIES,
        'fusion': 'rrf',
        'rrf_k': RRF_K,
        'candidate_pool': CONTEXT_POOL,
        'retry_pool_multiplier': RETRY_POOL_MULTIPLIER,
    },
    'prompts': {
        'rewrite_system': REWRITE_SYSTEM,
        'decompose_system': DECOMPOSE_SYSTEM,
        'route_system': ROUTE_SYSTEM,
    },
    'routing_rule': 'the router chooses how hard to search; only the generator may decide the evidence is insufficient',
    'abstention_sentinel': ABSTAIN_SENTINEL,
    'selection_rule': 'feasible (abstention accuracy == 1.0, citation validity == 1.0), then max work coverage, max answer rate, max grounding, min added calls, then simplicity',
    'frozen_pipeline': {                                   # the stage 05-08 machine this policy sits in front of
        'retrieval_method': SELECTED_METHOD,
        'reranker': SELECTED_RERANKER,
        'top_k': TOP_K,
        'context_builder': SELECTED_BUILDER,
        'token_budget': CONTEXT_TOKEN_BUDGET,
        'grounding_contract': SELECTED_CONTRACT,
    },
    'candidates': policy_metrics,                          # coverage, answer rate, abstention, citation, grounding, cost
    'evaluated_questions': {'answerable': answerable_count, 'unanswerable': unanswerable_count},
    'production_target': {
        'module': 'production/retrieval/query_processor.py',
        'entrypoint': 'process_query(query)',
        'returns': "{'normalized': str, 'variants': list[str], 'embedding': list[float]}",
    },
}
query_handoff_path.write_text(json.dumps(query_handoff, ensure_ascii=False, indent=2), encoding='utf-8')

print(f'Wrote decision record to {query_handoff_path.relative_to(ROOT)}')
print(f'Selected policy: {selected_policy} | parameters: {query_handoff["policy_parameters"]}')
print('Reminder: .artifacts/ is gitignored and notebook outputs are stripped on commit - do not commit the handoff.')


## Observations and Open Questions

Honest notes from working through this stage — what the numbers do and do not prove, and what to revisit later.

- **Query handling can only improve retrieval, never grounding.** Decomposition broadens the *pool* the context is built from, but the strict stage 08 contract still decides what the answer says and whether to abstain. That separation is deliberate: it is why a broader search can raise coverage without ever creating a new way to hallucinate.
- **Coverage is upstream of the budget.** Fusing several sub-query pools surfaces more works, but the `dedup-parent` builder still cuts to the top passages under the token budget. On a tight budget, coverage won by decomposition can be partly spent back at context building — a tension that would grow with more works per question.
- **The router is an optimizer, not a judge.** It picks how hard to search and whether a retry is worth attempting; it never declares a question answerable or not. Keeping that authority with the generator is what lets abstention accuracy stay a hard constraint even for the adaptive policy.
- **The bounded retry is a near-miss recoverer, not a safety net.** It only fires on an abstention the router thought was in-corpus, and it fires once. On a truly out-of-corpus question the generator abstains again after the broader search, so the retry cannot turn a correct refusal into a hallucination.
- **Small, single-turn eval set.** Six dev questions pre-screen policies; they cannot certify quality, and they exercise none of the conversational or trivial-query routing that a production system would. Stage 10's held-out judge and stage 12's safety work are where those gaps close.
- **Caching makes the comparison reproducible.** Reformulations and answers are replayed from disk, so the policy comparison is against a fixed baseline rather than a moving target, and re-running after a kernel restart bills nothing.


## Completion Check and Next Notebook

- **Restart and run in order:** restart the kernel and run top to bottom to confirm this notebook depends only on the notebook 04-08 artifacts, not hidden state. Query reformulation, embedding, and generation are all served from disk caches when warm, so a clean run bills nothing.
- **What we achieved:** four query-handling policies (passthrough, rewrite, decompose, adaptive) run in front of the frozen stage 05-08 pipeline and graded on work coverage, answer rate, abstention accuracy, citation validity, grounding, work relevance, added calls, and latency; a worked cross-play example showing how decomposition broadens coverage; a deterministic, constraint-gated selection that keeps the simplest policy which covers every expected work, preserves abstention and citations, and stays inside a small call budget; and the exported query handoff.
- **Honest limits:** the labeled dev set is small and single-turn, and grounding is a lexical proxy, so these numbers pre-screen policies rather than certify end-to-end quality. Conversational follow-up resolution is explained but not measured for lack of multi-turn data. The systematic judge is notebook 10.
- **Privacy and hygiene:** outputs are stripped on commit by `nbstripout`; `.artifacts/` is ignored. Do not commit the handoff or the query cache.
- **Next stage — 10 Continuous Evaluation:** load `.artifacts/query-handoff.json` and the frozen pipeline, then score the end-to-end system on the held-out questions with an Large Language Model (LLM) judge — faithfulness, answer relevance, and context relevance — turning the lexical proxies used so far into systematic, defensible evaluation.


## Questions, Discussions, and Further Insights

The "Observations and Open Questions" cell records caveats about *our* numbers. These questions build the transferable intuition.

**Q1. Why touch the query at all — isn't retrieval the stage that finds evidence?**
Retrieval only ever sees the string we hand it, and that string is written for a human. A single embedding of "how do the ghosts change the two heroes?" leans toward whichever play dominates the vector and quietly returns one side of a two-play comparison. Query handling is the stage that notices the question has *two* targets and gives retrieval two queries, so coverage becomes a property we engineer rather than hope for.

**Q2. Why is decomposition the main lever here rather than rewriting?**
Rewriting changes the *wording* of one query; decomposition changes the *number* of queries. Vague phrasing is a wording problem a rewrite can fix, but a cross-play comparison or an ambiguous name is a coverage problem no single query solves — the embedding cannot point at three Antonios at once. Decomposition plus multi-query fusion is the only policy structurally capable of covering every work, which is why work coverage is the metric that separates the arms.

**Q3. Why reuse Reciprocal Rank Fusion (RRF) again — didn't stage 05 already use it?**
RRF combines *rankings*, and it does not care whether those rankings came from different retrievers (stage 05's hybrid search) or from different sub-queries (this stage). Reusing it keeps one well-understood, parameter-light fusion rule across the pipeline: each sub-query contributes its ordered pool, and a passage ranked highly by any sub-query floats to the top of the fused list. Fewer moving parts, same intuition.

**Q4. Why must the router never decide an answer is unsupported?**
Because the router reads only the *question*, not the evidence. A cheap classifier guessing "this is out of corpus" would move the grounding decision away from the passages and back into a model's parametric memory — the exact failure stages 07 and 08 built the context and the strict contract to prevent. The router may choose how hard to search; only the generator, reading the retrieved passages, may decide whether the evidence supports an answer. That is why abstention accuracy is a hard constraint on every policy, including adaptive.

**Q5. Why cap the adaptive retry at one, instead of looping until an answer appears?**
An unbounded loop raises recall asymptotically while cost, latency, and non-determinism run away — two users asking the same question could get different amounts of work and different answers. A single bounded retry captures most of the recoverable cases (a near-miss that a broader search fixes) while keeping the per-query cost predictable and the behavior reproducible. Bounded beats clever.

**Q6. How can adding query handling ever make things worse?**
Three ways, all measured here. A rewrite can drift the intent and retrieve confidently for the wrong question. Over-eager decomposition can fragment a simple question and pull in loosely related passages, costing precision for coverage it did not need. And every policy spends extra calls, so one that matches a simpler policy on quality but costs more is strictly worse. The selection rule gates on safety, then rewards coverage, then penalizes cost — so a policy cannot "win" by doing more work for the same result.

**Q7. What did we produce, and what happens next?**
We exported `.artifacts/query-handoff.json` — the selected query-handling policy, its reproducible parameters and prompts, and the measured comparison that justified it. Stage 10 (Continuous Evaluation) replays this policy on top of the frozen pipeline and judges the end-to-end answers systematically on the held-out set; stages 11-12 serve the selected policy in production through `process_query`.
